# NSU BDA. 2024 Accidents
Соревнование для студенттов курса АБМД, ФИТ НГУ 2024

Студент: Митюшин Владимир 24221

In [551]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler

import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Input, BatchNormalization

Загрузим данные

In [552]:
X_train = pd.read_csv('../data/X_train.csv', index_col=0)
X_testFinal = pd.read_csv('../data/X_test.csv', index_col=0)
Y_train = pd.read_csv('../data/Y_train.csv', index_col=0)

Выведем информацию по датасетам

## Очистка и подготовка данных

In [553]:
df = X_train.copy()

In [554]:
drop_indexes = df.index[df['age_of_vehicle'] > 25].tolist()
# drop_indexes = drop_indexes + df.index[df['age_band_of_casualty'] == -1].tolist() + df.index[df['weather_conditions'] == -1].tolist() + df.index[df['road_surface_conditions'] == -1].tolist()
df.drop(drop_indexes, inplace=True)
Y_train.drop(drop_indexes, inplace=True)


df['sex_of_casualty'] = df['sex_of_casualty'].replace(9, -1).replace(-1, df['sex_of_casualty'].mean())
df['car_passenger'] = df['car_passenger'].replace(9, -1)
# df['casualty_type'] = df['casualty_type'].replace([3, 4, 5, 22, 97, 103, 104, 105, 106], 2).replace(113, 19).replace([9, 10, 11, 17, 18, 19, 20, 21, 23, 90, 97, 98], 8)
df['vehicle_type'] = df['vehicle_type'].replace([3, 4, 5, 22, 23, 97, 103, 104, 105, 106], 2).replace([20, 21, 98, 113], 19).replace([9, 10, 11, 108, 109, 110], 8).replace(99, 90)
df['vehicle_manoeuvre'] = df['vehicle_manoeuvre'].replace([8, 10], 3).replace([17, 18], 16).replace(99, -1)
df.describe()

,vehicle_reference,casualty_class,sex_of_casualty,age_of_casualty,age_band_of_casualty,pedestrian_location,pedestrian_movement,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,...,speed_limit,junction_detail,junction_control,second_road_class,second_road_number,pedestrian_crossing_human_control,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,...,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000
mean,1.457198,1.473239,1.385068,36.783498,6.307857,0.760861,0.615109,0.220797,0.050626,0.026024,...,37.519117,3.786146,1.679830,2.996640,223.640848,0.323232,1.107123,2.057349,1.639340,1.372062
std,2.328807,0.725779,0.484800,19.515151,2.453357,2.147320,1.965782,0.553609,0.437022,0.232720,...,14.734068,12.024490,2.516754,2.757978,936.349753,1.640876,2.375471,1.735270,1.791897,0.931780
min,1.000000,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,1.000000,1.000000,1.000000,22.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,30.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000,34.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,30.000000,1.000000,2.000000,3.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
75%,2.000000,2.000000,2.000000,50.000000,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,50.000000,3.000000,4.000000,6.000000,0.000000,0.000000,0.000000,4.000000,1.000000,2.000000
max,999.000000,3.000000,2.000000,102.000000,11.000000,10.000000,9.000000,2.000000,9.000000,2.000000,...,70.000000,99.000000,9.000000,6.000000,9999.000000,9.000000,9.000000,7.000000,9.000000,9.000000


In [555]:
# df['pedestrian_location'] = df['pedestrian_location'].replace(10, -1)
# df['bus_or_coach_passenger'] = df['bus_or_coach_passenger'].replace(-1, 0)
# df['pedestrian_road_maintenance_worker'] = df['pedestrian_road_maintenance_worker'].replace(-1, 0)
df['pedestrian_crossing_physical_facilities'] = df['pedestrian_crossing_physical_facilities'].replace(-1, 0)
# df['weather_conditions'] = df['weather_conditions'].replace(-1, 0)
# df['road_surface_conditions'] = df['road_surface_conditions'].replace(-1, 0)
# df['speed_limit'] = df['speed_limit'].replace(-1, df['speed_limit'].mean())

In [556]:
# df.describe()

In [557]:
# import holoviews as hv
# from holoviews import dim
# from holoviews import opts
# hv.extension('bokeh')


# def f(x):
#     return hv.BoxWhisker(df[x]).opts(height=120, responsive=True, toolbar='above', invert_axes=True, tools=['hover'])

# hv.DynamicMap(f, kdims=['x']).redim.values(x=df.select_dtypes('number').columns)

In [558]:
df.drop(columns=['accident_index', 'vehicle_reference'], inplace=True)
df = df.drop(columns=['age_of_casualty', 'vehicle_left_hand_drive', 'pedestrian_movement', 'first_road_number',
                      'second_road_number', 'junction_control', 'pedestrian_crossing_human_control']) # , 'casualty_home_area_type'

### Посмотрим на категориальные признаки

In [559]:
df = df.drop(columns='generic_make_model')

In [560]:
df['local_authority_highway'] = df['local_authority_highway'].str[:3]
display(df['local_authority_highway'].value_counts(dropna=False))

local_authority_highway
E10    158523
E06    115079
E09    100761
E08     83500
S12     21799
W06     16905
EHE        98
Name: count, dtype: int64

In [561]:
one_hot = pd.get_dummies(df.select_dtypes('O'), prefix=df.select_dtypes('O').columns, dtype=bool)
df = pd.concat([one_hot, df.select_dtypes('number'), df.select_dtypes('bool')], axis=1)

df.describe()

,casualty_class,sex_of_casualty,age_band_of_casualty,pedestrian_location,car_passenger,bus_or_coach_passenger,pedestrian_road_maintenance_worker,casualty_type,casualty_home_area_type,casualty_distance_banding,...,police_force,first_road_class,road_type,speed_limit,junction_detail,second_road_class,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,...,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000,496665.000000
mean,1.473239,1.385068,6.307857,0.760861,0.220797,0.050626,0.026024,7.624205,1.066266,1.585238,...,28.313888,4.144238,5.232448,37.519117,3.786146,2.996640,1.114697,2.057349,1.639340,1.372062
std,0.725779,0.484800,2.453357,2.147320,0.553609,0.437022,0.232720,10.534287,0.918715,1.423882,...,24.407225,1.470678,1.673324,14.734068,12.024490,2.757978,2.370329,1.735270,1.791897,0.931780
min,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,1.000000,1.000000,1.000000,-1.000000,-1.000000,-1.000000,0.000000,-1.000000,-1.000000,-1.000000
25%,1.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,...,5.000000,3.000000,6.000000,30.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
50%,1.000000,1.000000,6.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,1.000000,...,23.000000,4.000000,6.000000,30.000000,1.000000,3.000000,0.000000,1.000000,1.000000,1.000000
75%,2.000000,2.000000,8.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,2.000000,...,45.000000,6.000000,6.000000,50.000000,3.000000,6.000000,0.000000,4.000000,1.000000,2.000000
max,3.000000,2.000000,11.000000,10.000000,2.000000,9.000000,2.000000,99.000000,3.000000,5.000000,...,99.000000,6.000000,9.000000,70.000000,99.000000,6.000000,9.000000,7.000000,9.000000,9.000000


## Обучение

### Подготовка данных

In [562]:
Y_all = Y_train.copy()
Y_1 = Y_all.copy()['casualty_severity'].replace(3, 2)
Y_2 = Y_all.copy()['casualty_severity'].replace(1, 2)
Y_2 = Y_2.apply(lambda y: y-2)
Y_1 = Y_1.apply(lambda y: y-1)

print(np.unique(Y_1), np.unique(Y_2))

[0 1] [0 1]


In [563]:
scaler = StandardScaler()

X_train, X_test, y_train, y_test = train_test_split(df, Y_2, random_state=42)
X_train = scaler.fit_transform(X_train, y_train)
X_test = scaler.transform(X_test)

y_train = keras.utils.to_categorical(y_train, num_classes=2)
y_test = keras.utils.to_categorical(y_test, num_classes=2)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

y_train shape: (372498, 2)
y_test shape: (124167, 2)


In [564]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(df, Y_1, random_state=42)
X_train1 = scaler.fit_transform(X_train1, y_train1)
X_test1 = scaler.transform(X_test1)

y_train1 = keras.utils.to_categorical(y_train1, num_classes=2)
y_test1 = keras.utils.to_categorical(y_test1, num_classes=2)
print('y_train shape:', y_train1.shape)
print('y_test shape:', y_test1.shape)
print(X_train1.shape, y_train1.shape)

y_train shape: (372498, 2)
y_test shape: (124167, 2)
(372498, 31) (372498, 2)


### Подготовка модели

In [565]:
from keras.api.optimizers import Adam

def create_model(shape, seed):
    keras.utils.set_random_seed(seed)
    model = Sequential() 
    model.add(Input(shape=(shape, 1)))
    model.add(Flatten())
    model.add(BatchNormalization())
    
    # model.add(Dense(440, activation='leaky_relu')) # 'relu', 'leaky_relu', 'elu', 'selu'
    # model.add(Dense(200, activation='leaky_relu')) # 'relu', 'leaky_relu', 'elu'  220
    # model.add(Dropout(.2))
    # model.add(Dense(240, activation='leaky_relu')) # 'relu', 'leaky_relu', 'elu'
    # model.add(Dense(90, activation='leaky_relu'))
    # model.add(Dropout(.15))
    model.add(Dense(round(shape/2), activation='sigmoid'))
    model.add(Dense(40, activation='leaky_relu'))
    model.add(Dense(40, activation='leaky_relu'))
    model.add(Dense(40, activation='leaky_relu'))
    # model.add(Dense(16, activation='leaky_relu'))
    model.add(Dense(2, activation='softmax')) # 'sigmoid', 'softmax', 'tanh'
    
    model.compile(optimizer=Adam(amsgrad=True),
                  loss='categorical_crossentropy', metrics=['f1_score'])
    return model

In [566]:
from keras.api.optimizers import Adam

def create_model_2(shape, seed):
    keras.utils.set_random_seed(seed)
    model = Sequential() 
    model.add(Input(shape=(shape, 1)))
    model.add(Flatten())
    model.add(BatchNormalization())
    
    # model.add(Dense(round(shape/2), activation='elu'))
    # # model.add(Dense(200, activation='leaky_relu')) # 'relu', 'leaky_relu', 'elu'
    # model.add(Dense(90, activation='leaky_relu'))
    # model.add(Dropout(.5))
    # model.add(Dense(36, activation='leaky_relu'))
    # # model.add(Dense(16, activation='leaky_relu'))
    # model.add(Dense(2, activation='softmax')) # 'sigmoid', 'softmax', 'tanh'

    model.add(Dense(440, activation='leaky_relu')) # 'relu', 'leaky_relu', 'elu', 'selu'
    model.add(Dropout(.5))
    model.add(Dense(140, activation='leaky_relu')) # 'relu', 'leaky_relu', 'elu'
    model.add(Dense(90, activation='leaky_relu'))
    model.add(Dropout(.25))
    model.add(Dense(16, activation='leaky_relu'))
    model.add(Dense(2, activation='softmax')) # 'sigmoid', 'softmax', 'tanh'
    
    model.compile(optimizer=Adam(amsgrad=True),
                  loss='categorical_crossentropy', metrics=['f1_score'])
    return model

In [567]:
# from keras.api.callbacks import EarlyStopping
# from scikeras.wrappers import KerasClassifier
# from sklearn.model_selection import cross_val_score
# from sklearn.model_selection import KFold
# from bayes_opt import BayesianOptimization
# import warnings
# warnings.filterwarnings('ignore')

# def nn_cl_bo2(class1, class2):
#     class_weights = [{0:class1, 1:class2}, {0:class2, 1:class1}]
    
#     es = EarlyStopping(monitor='val_loss', mode='min', verbose=0, patience=10)
#     nn = KerasClassifier(model=create_model(142), epochs=70, batch_size=4200, verbose=0, class_weight=class_weights)
#     kfold = KFold(n_splits=3, shuffle=True, random_state=123)
#     score = cross_val_score(nn, X_train, y_train, scoring='f1_macro', cv=kfold, fit_params={'callbacks':[es]})
#     score = np.nan_to_num(score)
#     score = np.max(score) 
#     return score

# params_nn2 ={
# 'class1': (0.5, 3.0),
# 'class2': (0.3, 1.0)
# }

# nn_bo = BayesianOptimization(nn_cl_bo2, params_nn2, random_state=42)
# nn_bo.maximize(init_points=25, n_iter=4)
# best_params = nn_bo.max['params']
# best_accuracy = nn_bo.max['target']

# print(f"Best Hyperparameters: {best_params}")
# print(f"Best Cross-Validation Accuracy: {best_accuracy}")

In [568]:
# def nn_cl_bo1(class1, class2):
#     class_weights = [{0:class1, 1:class2}, {0:class2, 1:class1}]
    
#     es = EarlyStopping(monitor='val_loss', mode='min', verbose=0, patience=10)
#     nn = KerasClassifier(model=create_model(242), epochs=15, batch_size=4200, verbose=0, class_weight=class_weights)
#     kfold = KFold(n_splits=3, shuffle=True, random_state=123)
#     score = cross_val_score(nn, X_train1, y_train1, scoring='f1_macro', cv=kfold, fit_params={'callbacks':[es]})
#     score = np.nan_to_num(score)
#     score = np.max(score) 
#     return score

# params_nn1 = {
#     'class1': (5, 10.0),
#     'class2': (0.5, 1.0)
# }

# nn_bo1 = BayesianOptimization(nn_cl_bo1, params_nn1, random_state=42)
# nn_bo1.maximize(init_points=25, n_iter=4)
# best_params1 = nn_bo1.max['params']
# best_accuracy1 = nn_bo1.max['target']
# print(f"Best Hyperparameters: {best_params1}")
# print(f"Best Cross-Validation Accuracy: {best_accuracy1}")

### Обучение модели

In [569]:
model = create_model(X_train.shape[1], 142)

class_weights = {
    0: 1.31, # 1.31
    1: 0.53  # 0.53
}

history = model.fit(X_train, y_train,
                    epochs=20,
                    validation_data=(X_test, y_test),
                    batch_size=1000,
                    class_weight=class_weights,
                    verbose=1
                    )

Epoch 1/20
373/373 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - f1_score: 0.5419 - loss: 0.4334 - val_f1_score: 0.6175 - val_loss: 0.5214
Epoch 2/20
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - f1_score: 0.6214 - loss: 0.4005 - val_f1_score: 0.6227 - val_loss: 0.5197
Epoch 3/20
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - f1_score: 0.6252 - loss: 0.3977 - val_f1_score: 0.6251 - val_loss: 0.5168
Epoch 4/20
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - f1_score: 0.6273 - loss: 0.3965 - val_f1_score: 0.6272 - val_loss: 0.5122
Epoch 5/20
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - f1_score: 0.6283 - loss: 0.3958 - val_f1_score: 0.6276 - val_loss: 0.5091
Epoch 6/20
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - f1_score: 0.6290 - loss: 0.3952 - val_f1_score: 0.6282 - val_loss: 0.5070
Epoch 7/20
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - f1_score: 0.6302 - loss: 0.3947 - val_f1_score: 0.6292 - val_loss: 0.5055
Epoch 8/20
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - f1_score: 0.6311 - loss: 0.3943 - val_f1_score: 0.

In [570]:
model2 = create_model_2(X_train1.shape[1], 242)

class_weights1 = {
    0: 8.25, # 8.3
    1: 0.69  # 0.69
}

history2 = model2.fit(X_train1, y_train1,
                    epochs=20,
                    validation_data=(X_test1, y_test1),
                    batch_size=700,
                    class_weight=class_weights1,
                    verbose=1
                    )

Epoch 1/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - f1_score: 0.5362 - loss: 0.2592 - val_f1_score: 0.5606 - val_loss: 0.1393
Epoch 2/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - f1_score: 0.5598 - loss: 0.2319 - val_f1_score: 0.5635 - val_loss: 0.1316
Epoch 3/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - f1_score: 0.5662 - loss: 0.2276 - val_f1_score: 0.5678 - val_loss: 0.1285
Epoch 4/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - f1_score: 0.5661 - loss: 0.2264 - val_f1_score: 0.5700 - val_loss: 0.1262
Epoch 5/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - f1_score: 0.5693 - loss: 0.2238 - val_f1_score: 0.5681 - val_loss: 0.1262
Epoch 6/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - f1_score: 0.5683 - loss: 0.2228 - val_f1_score: 0.5739 - val_loss: 0.1266
Epoch 7/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - f1_score: 0.5690 - loss: 0.2210 - val_f1_score: 0.5718 - val_loss: 0.1250
Epoch 8/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - f1_score: 0.5699 - loss: 0.2216 - val_f1_score: 0.

## Предсказание

In [571]:
X_testFinal = pd.read_csv('../data/X_test.csv', index_col=0)

X_testFinal = X_testFinal.reindex(df.columns, axis=1, fill_value=0)
X_testFinal_data = scaler.transform(X_testFinal)
X_testFinal.describe()

,local_authority_highway_E06,local_authority_highway_E08,local_authority_highway_E09,local_authority_highway_E10,local_authority_highway_EHE,local_authority_highway_S12,local_authority_highway_W06,casualty_class,sex_of_casualty,age_band_of_casualty,...,police_force,first_road_class,road_type,speed_limit,junction_detail,second_road_class,pedestrian_crossing_physical_facilities,light_conditions,weather_conditions,road_surface_conditions
count,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.0,166352.000000,166352.000000,166352.000000,...,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000,166352.000000
mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.470322,1.366873,6.305587,...,28.323892,4.149598,5.238975,37.547730,3.767667,3.008446,1.095220,2.051848,1.634257,1.371369
std,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.722688,0.532466,2.455111,...,24.336775,1.469986,1.665436,14.710353,11.917428,2.759564,2.362987,1.732798,1.787258,0.933177
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,-1.000000,-1.000000,...,1.000000,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,1.000000,5.000000,...,5.000000,3.000000,6.000000,30.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
50%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,1.000000,6.000000,...,23.000000,4.000000,6.000000,30.000000,2.000000,3.000000,0.000000,1.000000,1.000000,1.000000
75%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.000000,2.000000,8.000000,...,45.000000,6.000000,6.000000,50.000000,3.000000,6.000000,0.000000,4.000000,1.000000,2.000000
max,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.000000,9.000000,11.000000,...,99.000000,6.000000,9.000000,70.000000,99.000000,6.000000,9.000000,7.000000,9.000000,9.000000


In [572]:
import random

y1_final = model.predict(X_testFinal_data)
y2_final = model2.predict(X_testFinal_data)
tmp1 = []
tmp2 = []
for i in range(len(y1_final)):
    tmp1.append(np.argmax(y1_final[i])+2)
    tmp2.append(np.argmax(y2_final[i])+1)

y_final = [random.randint(1, 3) for i in range(X_testFinal_data.shape[0])]
for i in range(X_testFinal_data.shape[0]):
    if tmp2[i] == 1:
        y_final[i] = 1
    else: 
        y_final[i] = tmp1[i]

print(np.unique(y_final))

5199/5199 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step
5199/5199 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step
[1 2 3]


In [573]:
my_file = open("../predicts/nn_balance36.csv", "w+")

my_file.write("Id,casualty_severity\n")
my_i = 0
for i, row in X_testFinal.iterrows():
    my_file.write(f"{i}, {y_final[my_i]}\n")
    my_i = my_i+1
my_file.close()